# Task 64: CNN on the CIFAR-10 dataset

Reference: https://www.analyticsvidhya.com/blog/2020/02/learn-image-classification-cnn-convolutional-neural-networks-3-datasets/

Default run uses 1,000 actual training images and 300 images from the original test split; set USE_FULL_DATASET=True for the full dataset.

In [1]:
# Task 64: CNN on the CIFAR-10 dataset
# Reference: https://www.analyticsvidhya.com/blog/2020/02/learn-image-classification-cnn-convolutional-neural-networks-3-datasets/

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import numpy as np
import tensorflow as tf

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(13)
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.experimental.enable_op_determinism()

from pathlib import Path

# The download includes a real CIFAR-10 sample in data/cifar10_sample.npz.
# Use the complete official dataset by setting USE_FULL_DATASET = True.
USE_FULL_DATASET = False
if USE_FULL_DATASET:
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    y_train, y_test = y_train.ravel(), y_test.ravel()
else:
    base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
    sample_path = base / 'data' / 'cifar10_sample.npz'
    if not sample_path.exists():
        raise FileNotFoundError('Keep data/cifar10_sample.npz beside these task files, '
                                'or enable USE_FULL_DATASET.')
    with np.load(sample_path, allow_pickle=False) as sample:
        x_train, y_train = sample['x_train'], sample['y_train']
        x_test, y_test = sample['x_test'], sample['y_test']

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0
model = tf.keras.Sequential([
    tf.keras.Input(shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(8, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10)
])
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
rng = np.random.default_rng(13)
print('Dataset:', 'Full CIFAR-10' if USE_FULL_DATASET else 'Real CIFAR-10 teaching subset')
print('Training images:', x_train.shape)
print('Test images:', x_test.shape)
print('Classes:', classes)
print('Trainable parameters:', model.count_params())

def train_epoch(model, optimizer, x, y, order, batch_size=64):
    losses = []
    for start in range(0, len(order), batch_size):
        indices = order[start:start + batch_size]
        x_batch = tf.convert_to_tensor(x[indices], dtype=tf.float32)
        y_batch = tf.convert_to_tensor(y[indices], dtype=tf.int64)
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                y_batch, logits, from_logits=True))
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        losses.append(float(loss.numpy()))
    return float(np.mean(losses))


def evaluate(model, x, y, batch_size=128):
    logits = []
    for start in range(0, len(x), batch_size):
        logits.append(model(tf.convert_to_tensor(x[start:start + batch_size]),
                            training=False).numpy())
    logits = np.concatenate(logits)
    predictions = np.argmax(logits, axis=1)
    loss = float(tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
        y, logits, from_logits=True)).numpy())
    accuracy = float(np.mean(predictions == y))
    return loss, accuracy, predictions

for epoch in range(1, 6):
    loss = train_epoch(model, optimizer, x_train, y_train, rng.permutation(len(x_train)))
    print(f'Epoch {epoch}: training loss={loss:.4f}')
loss, accuracy, predictions = evaluate(model, x_test, y_test)
print('Held-out test loss:', round(loss, 4))
print('Held-out test accuracy:', round(accuracy, 4))
print('First five predictions:')
for actual, predicted in zip(y_test[:5], predictions[:5]):
    print('Actual:', classes[int(actual)], '| Predicted:', classes[int(predicted)])
print('A small teaching subset and five epochs are not a full benchmark.')


Dataset: Real CIFAR-10 teaching subset
Training images: (1000, 32, 32, 3)
Test images: (300, 32, 32, 3)
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Trainable parameters: 34522
Epoch 1: training loss=2.3051
Epoch 2: training loss=2.2649
Epoch 3: training loss=2.2018
Epoch 4: training loss=2.1144
Epoch 5: training loss=2.0118
Held-out test loss: 2.0258
Held-out test accuracy: 0.24
First five predictions:
Actual: cat | Predicted: cat
Actual: ship | Predicted: automobile
Actual: ship | Predicted: ship
Actual: airplane | Predicted: ship
Actual: frog | Predicted: dog
A small teaching subset and five epochs are not a full benchmark.
